# Semana 3. Introducción a redes neuronales convolucionales con TensorFlow


**Objetivos:** (1) explicar el cálculo de una convolución; (2) construir un pipeline de datos seguro; (3) distinguir filtros, mapas de características y atribuciones; (4) comparar convolución 1D y dilatada en series de tiempo.


## 0. Preparación y reproducibilidad
Una imagen tiene forma `(alto, ancho, canales)`. Un batch tiene forma `(N, alto, ancho, canales)`; `Conv2D`.

In [ ]:
import os, pathlib, numpy as np, pandas as pd, matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, ConfusionMatrixDisplay
SEED = 42
tf.keras.utils.set_random_seed(SEED)
AUTOTUNE = tf.data.AUTOTUNE
print('TensorFlow:', tf.__version__, '| GPUs:', tf.config.list_physical_devices('GPU'))

## 1. Lectura del dataset y *data slices*
entrenamiento (70%), validación (15%) y prueba (15%). No se consulta test durante el entrenamiento.

In [ ]:
(raw_train, raw_val, raw_test), ds_info = tfds.load(
    'cats_vs_dogs', split=['train[:20%]', 'train[70%:75%]', 'train[95%:]'],
    as_supervised=True, shuffle_files=True, with_info=True)
class_names = ds_info.features['label'].names
print('Clases:', class_names)
print('Ejemplo de especificación:', raw_train.element_spec)
for name, ds in [('train', raw_train), ('validation', raw_val), ('test', raw_test)]:
    print(name, tf.data.experimental.cardinality(ds).numpy())

In [ ]:

fig, axes = plt.subplots(2, 4, figsize=(12, 7))
for ax, (img, label) in zip(axes.ravel(), raw_train.take(8)):
    ax.imshow(img.numpy())
    ax.set_title(f'{class_names[int(label)]}\n{tuple(img.shape)}')
    ax.axis('off')
plt.tight_layout(); plt.show()


sample_labels = [int(y.numpy()) for _, y in raw_train.take(500)]
print(pd.Series(sample_labels).map(dict(enumerate(class_names))).value_counts())

### Discusión: una operación de convolución
Para un canal de salida y un desplazamiento `(i,j)`, la operación (correlación cruzada) es:

$Y_{i,j,k}=b_k+\sum_{u,v,c} W_{u,v,c,k}X_{i+u,j+v,c}.$

Por ejemplo, un `Conv2D(16, kernel_size=5, padding="same")` aplicado a `(128,128,3)` devuelve `(128,128,16)` y tiene `5×5×3×16+16 = 1216` parámetros. `MaxPooling2D(2)` reduce las dimensiones espaciales a la mitad.

tf.keras.layers.Conv2D( \
    filters,\
    kernel_size, \
    strides=(1, 1), \
    padding='valid',\
    data_format=None,\
    dilation_rate=(1, 1),\
    groups=1,
    activation=None,\
    use_bias=True,\
    kernel_initializer='glorot_uniform',\
    bias_initializer='zeros', \
    kernel_regularizer=None,\
    bias_regularizer=None,\
)


In [ ]:
# Experimento: forma de entrada/salida y número de parámetros
mini_layer = tf.keras.layers.Conv2D(16, 5, padding='same')
example = tf.zeros((1, 128, 128, 3))
print('Entrada:', example.shape, 'Salida:', mini_layer(example).shape)
print('Parámetros:', mini_layer.count_params())

## 2. Preprocesamiento con `tf.data`

In [ ]:
IMAGE_SIZE = (128, 128)
BATCH_SIZE = 32

def preprocess(image, label):
    image = tf.image.resize(image, IMAGE_SIZE)
    image = tf.cast(image, tf.float32) / 255.0
    return image, tf.cast(label, tf.int32)

def augment(image, label):
    image = tf.image.random_brightness(image, max_delta=0.12)
    image = tf.image.random_contrast(image, lower=0.85, upper=1.15)
    image = tf.clip_by_value(image, 0.0, 1.0)
    return image, label

def make_dataset(raw, training=False):
    ds = raw.map(preprocess, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.shuffle(2048, seed=SEED, reshuffle_each_iteration=True)
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_dataset(raw_train, training=True)
val_ds = make_dataset(raw_val)
test_ds = make_dataset(raw_test)
for x_batch, y_batch in train_ds.take(1):
    print('x:', x_batch.shape, x_batch.dtype, '| y:', y_batch.shape, y_batch.dtype)
    print('Rango de píxeles:', float(tf.reduce_min(x_batch)), float(tf.reduce_max(x_batch)))

In [ ]:
images, labels = next(iter(train_ds))
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for k, ax in enumerate(axes.ravel()):
    ax.imshow(images[k].numpy())
    ax.set_title(class_names[int(labels[k])]); ax.axis('off')
plt.tight_layout(); plt.show()

## 3. Arquitectura CNN 
Se mantiene el patrón original `Conv2D → MaxPooling2D` tres veces. Para que Grad-CAM sea directo y para reducir parámetros, se usa `GlobalAveragePooling2D` en lugar de `Flatten`. La capa `Dense` final es la cabeza de clasificación de la CNN.

In [ ]:


inputs = tf.keras.Input(shape=(*IMAGE_SIZE, 3), name='imagen')
x = layers.Conv2D(16, 5, padding='same', activation='relu', name='conv_1')(inputs)
x = layers.MaxPooling2D(2, name='pool_1')(x)
x = layers.Conv2D(32, 3, padding='same', activation='relu', name='conv_2')(x)
x = layers.MaxPooling2D(2, name='pool_2')(x)
x = layers.Conv2D(64, 3, padding='same', activation='relu', name='conv_3')(x)
x = layers.MaxPooling2D(2, name='pool_3')(x)
x = layers.GlobalAveragePooling2D(name='global_avg_pool')(x)
x = layers.Dropout(0.25, name='dropout')(x)
outputs = layers.Dense(2, activation='softmax', name='probabilidades')(x)
model =tf.keras.Model(inputs, outputs, name='CNN_Gatos_Perros')
model.summary()
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
              loss=tf.keras.losses.SparseCategoricalCrossentropy(),
              metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name='accuracy')])

## 4. Entrenamiento y callbacks

In [ ]:
OUT_DIR = pathlib.Path('resultados_clase7_cnn')
OUT_DIR.mkdir(exist_ok=True)
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=4,
                                  restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                     patience=2, min_lr=1e-6),
    tf.keras.callbacks.ModelCheckpoint(str(OUT_DIR / 'mejor_cnn.keras'),
                                    monitor='val_loss', save_best_only=True),
    tf.keras.callbacks.TensorBoard(log_dir=str(OUT_DIR / 'logs'),write_graph=True,histogram_freq=1,write_images=True),
]
EPOCHS = 5
history = model.fit(train_ds, validation_data=val_ds,
                    epochs=EPOCHS, callbacks=callbacks, verbose=1)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
for ax, key in zip(axs, ['loss', 'accuracy']):
    ax.plot(history.history[key], label='entrenamiento')
    ax.plot(history.history['val_' + key], label='validación')
    ax.set(xlabel='Época', ylabel=key, title=key)
    ax.legend(); ax.grid(alpha=.25)
plt.tight_layout(); plt.show()
# tensorboard --logdir resultados_clase7_cnn/logs

## 5. Evaluación final en test
Se calculan la pérdida y exactitud, matriz de confusión, precisión/recall/F1, ROC-AUC y se inspeccionan ejemplos correctos y fallidos. **Solo ahora** se utiliza el conjunto de prueba.

In [ ]:
results = model.evaluate(test_ds, return_dict=True)
print('Métricas test:', results)
y_true = np.concatenate([yb.numpy() for _, yb in test_ds])
pred_prob = model.predict(test_ds, verbose=0)
y_pred = np.argmax(pred_prob, axis=1)
print(classification_report(y_true, y_pred, target_names=class_names, digits=3))
print('ROC-AUC:', roc_auc_score(y_true, pred_prob[:, 1]))
cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(cmap='Blues', values_format='d')
plt.title('Matriz de confusión — test'); plt.show()

In [ ]:
# Comparar probabilidades en unas pocas imágenes del conjunto de prueba
sample_x, sample_y = next(iter(test_ds))
sample_prob = model.predict(sample_x, verbose=0)
fig, axs = plt.subplots(2, 4, figsize=(12, 6))
for k, ax in enumerate(axs.ravel()):
    ax.imshow(sample_x[k])
    idx = int(np.argmax(sample_prob[k]))
    ax.set_title(f'Real: {class_names[int(sample_y[k])]}\nPred: {class_names[idx]} ({sample_prob[k,idx]:.2f})')
    ax.axis('off')
plt.tight_layout(); plt.show()

## 6. ¿Qué detectan las convoluciones?
**visualizaciones complementarias:** (A) un kernel aprendido, (B) sus mapas de características para una imagen.

In [ ]:
# A. Pesos de la primera convolución: [alto, ancho, canal entrada, canal salida]
kernels, biases = model.get_layer('conv_1').get_weights()
print('Pesos:', kernels.shape, '| sesgos:', biases.shape)
fig, axs = plt.subplots(2, 4, figsize=(10, 6))
for k, ax in enumerate(axs.ravel()):
    kernel = kernels[:, :, :, k]
    kernel = (kernel - kernel.min()) / (np.ptp(kernel) + 1e-8)
    im=ax.imshow(kernel, cmap='viridis'); ax.set_title(f'Filtro {k}'); ax.axis('off')
cbar_ax = fig.add_axes([0.30, 0.04, 0.40, 0.025]) 
fig.colorbar(im, cax=cbar_ax, orientation='horizontal')
plt.suptitle('Kernels aprendidos en la primera capa'); plt.tight_layout(); plt.show()

In [ ]:
# B. Visualizar salida de capas convolucionales y de pooling
feature_model = tf.keras.Model(model.input,
                            [model.get_layer(name).output for name in
                             ['conv_1','pool_1','conv_2','pool_2','conv_3','pool_3']])
img_one = sample_x[0:1]
feature_maps = feature_model.predict(img_one, verbose=0)
fig, axs = plt.subplots(len(feature_maps), 5, figsize=(12, 14))
for row, (name, fmap) in enumerate(zip(['conv_1','pool_1','conv_2','pool_2','conv_3','pool_3'],feature_maps)):
    for k in range(5):
        axs[row,k].imshow(fmap[0, :, :, k], cmap='viridis')
        axs[row,k].axis('off')
        if k == 0: axs[row,k].set_title(f'{name}\n{fmap.shape[1:]}', fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# C. Demostración de un kernel de bordes verticales (NO entrenado)
gray = tf.image.rgb_to_grayscale(img_one)  # (1,128,128,1)
edge_kernel = tf.constant([[-1.,0.,1.],[-2.,0.,2.],[-1.,0.,1.]], dtype=tf.float32)
edge_kernel = edge_kernel[:, :, tf.newaxis, tf.newaxis]
edges = tf.nn.conv2d(gray, edge_kernel, strides=1, padding='SAME')
fig, ax = plt.subplots(1, 3, figsize=(11, 4))
ax[0].imshow(img_one[0]); ax[0].set_title('Imagen original')
ax[1].imshow(gray[0,:,:,0], cmap='gray'); ax[1].set_title('Escala de grises')
ax[2].imshow(edges[0,:,:,0], cmap='gray'); ax[2].set_title('Respuesta al kernel Sobel')
for a in ax:a.axis('off')
plt.tight_layout(); plt.show()

## 7. Grad-CAM: ¿dónde miró la CNN?
Grad-CAM calcula gradientes de la puntuación de clase con respecto al último tensor convolucional, promedia esos gradientes por canal y pondera los mapas de activación. **Es una explicación local aproximada**, no una prueba causal de que la región sea necesaria para la predicción.

$\alpha_k^c=\frac{1}{HW}\sum_{i,j}\frac{\partial s_c}{\partial A^k_{ij}},\qquad L^c=\mathrm{ReLU}\left(\sum_k\alpha_k^c A^k
\right).$

Aquí `s_c` es la probabilidad de la clase; para aplicaciones más rigurosas se pueden usar logits antes de softmax.

In [ ]:
def grad_cam_heatmap(model, image_batch, class_index=None, layer_name='conv_3'):
    # image_batch: (1,H,W,3), normalizada
    extractor = tf.keras.Model(model.inputs, [model.get_layer(layer_name).output, model.output])
    with tf.GradientTape() as tape:
        conv_outputs, predictions = extractor(image_batch, training=False)
        if class_index is None:
            class_index = int(tf.argmax(predictions[0]))
        score = predictions[:, class_index]
    grads = tape.gradient(score, conv_outputs)
    weights = tf.reduce_mean(grads, axis=(1, 2), keepdims=True)
    cam = tf.reduce_sum(weights * conv_outputs, axis=-1)
    cam = tf.nn.relu(cam)[0]
    cam = cam / (tf.reduce_max(cam) + 1e-8)
    cam = tf.image.resize(cam[..., tf.newaxis], IMAGE_SIZE).numpy()[:,:,0]
    return cam, class_index

cam, target = grad_cam_heatmap(model, img_one)
fig, ax = plt.subplots(1, 3, figsize=(12,4))
ax[0].imshow(img_one[0]); ax[0].set_title('Imagen')
ax[1].imshow(cam, cmap='jet', vmin=0, vmax=1); ax[1].set_title('Mapa Grad-CAM')
ax[2].imshow(img_one[0]); ax[2].imshow(cam, cmap='jet', alpha=.45, vmin=0, vmax=1)
ax[2].set_title(f'Clase: {class_names[target]}')
for a in ax:a.axis('off')
plt.tight_layout(); plt.show()

## 8. SHAP: atribución local de píxeles
SHAP aproxima contribuciones de características respecto a un **conjunto de referencia**. A diferencia de Grad-CAM, produce atribuciones a nivel de píxel/canal. En CNN grandes es computacionalmente costoso y sensible al fondo elegido.

Usamos `shap.GradientExplainer` con una pequeña muestra de entrenamiento como referencia y una imagen de test como ejemplo. Los valores positivos/negativos describen la dirección de la contribución **respecto al fondo**, no evidencia causal. Reducir `nsamples` para demostraciones rápidas.

In [ ]:
# Descomentar si no está instalado:
# %pip install shap
import shap
background_x = next(iter(train_ds))[0][:146].numpy().astype('float32')
explain_x = sample_x[0:1].numpy().astype('float32')
explainer = shap.GradientExplainer(model, background_x)
shap_values = explainer.shap_values(explain_x, nsamples=400)
print('Tipo:', type(shap_values))
print('Formas:', [v.shape for v in shap_values] if isinstance(shap_values, list) else np.asarray(shap_values).shape)

In [ ]:
# Compatible con retorno lista [clases] o array (N,H,W,C,clases)
target=sample_y[:1]
if isinstance(shap_values, list):
    attr = np.asarray(shap_values[target])[0]
else:
    vals = np.asarray(shap_values)
    attr = vals[0, ..., target] if vals.ndim == 5 else vals[0]
attr_2d = np.sum(attr, axis=-1) if attr.ndim == 3 else attr
lim = np.quantile(np.abs(attr_2d), 0.99) + 1e-9
fig, axs = plt.subplots(1, 3, figsize=(13,4))
axs[0].imshow(explain_x[0]); axs[0].set_title('Imagen')
m = axs[1].imshow(attr_2d[0]*5000, cmap='seismic', vmin=-lim, vmax=lim) ## Escala erronea!!! mas entrenamiento 
axs[1].set_title(f'SHAP: {class_names[target.numpy()[0]]}'); plt.colorbar(m, ax=axs[1], fraction=.046)
axs[2].imshow(explain_x[0]); axs[2].imshow(attr_2d[0]*5000, cmap='seismic', alpha=.55, vmin=-lim, vmax=lim)
axs[2].set_title('Superposición')
for a in axs:a.axis('off')
plt.tight_layout(); plt.show()
print('Interpretación: rojo aumenta y azul disminuye la salida de la clase vs. fondo.')

In [ ]:
print('attr min/max:', attr.min(), attr.max())
print('attr_2d min/max:', attr_2d.min(), attr_2d.max())

## 9. Series de tiempo: manchas solares 

**Problema supervisado:** de los valores `[t-n, …, t-1]`, predecir el valor en `t`.

In [ ]:
lpath = pathlib.Path(r'C:\Users\javier\Documents\deep_learning_2026_phd\deep-learning-i\clase1\sunspots.csv')
filepath = str(lpath)
df = pd.read_csv(filepath)
print(df.head(), df.columns.tolist())

value_col = 'Monthly Mean Total Sunspot Number'
assert value_col in df.columns, f'No se encuentra {value_col}. Columnas: {df.columns.tolist()}'
series = df[value_col].to_numpy(dtype='float32')
dates = pd.to_datetime(df['Date'])
plt.figure(figsize=(13,3)); plt.plot(dates, series); plt.title('Manchas solares mensuales'); plt.show()

In [ ]:
WINDOW_SIZE = 60
BATCH_TS = 32
n = len(series)
train_end = int(n * 0.70)
val_end = int(n * 0.85)
mu, sigma = float(series[:train_end].mean()), float(series[:train_end].std())
sigma = max(sigma, 1e-6)
series_norm = (series - mu) / sigma
print('train / valid / test:', train_end, val_end-train_end, n-val_end)

def make_windows(full_values, target_start, target_end, window=WINDOW_SIZE):
    targets = np.arange(max(target_start, window), target_end)
    X = np.stack([full_values[t-window:t] for t in targets], axis=0)[..., np.newaxis]
    y = full_values[targets, np.newaxis]
    return X.astype('float32'), y.astype('float32'), targets

X_train, y_train, idx_train = make_windows(series_norm, 0, train_end)
X_val, y_val, idx_val = make_windows(series_norm, train_end, val_end)
X_test, y_test, idx_test = make_windows(series_norm, val_end, n)

def to_ts_dataset(X, y, training=False):
    ds = tf.data.Dataset.from_tensor_slices((X,y))
    if training: 
        ds = ds.shuffle(1024, seed=SEED)
    return ds.batch(BATCH_TS).prefetch(AUTOTUNE)
train_ts = to_ts_dataset(X_train, y_train, True)
val_ts = to_ts_dataset(X_val, y_val)
test_ts = to_ts_dataset(X_test, y_test)
print('Ventanas:', X_train.shape, X_val.shape, X_test.shape)

## 10. Una arquitectura con Conv1D
Una convolución `padding="causal"`. Como la tarea es predecir el siguiente valor a partir de toda la ventana pasada, `GlobalAveragePooling1D` resume las características temporales antes de la salida escalar.

**Experimento:** `Conv1D(filters=32, kernel_size=3)` → `Conv1D(32,3)` → `GlobalAveragePooling1D` → `Dense(1)`. 

In [ ]:
def build_conv1d():
    inp = tf.keras.Input(shape=(WINDOW_SIZE, 1))
    x = layers.Conv1D(32, 3, padding='causal', activation='relu')(inp)
    x = layers.Conv1D(32, 3, padding='causal', activation='relu')(x)
    x = layers.GlobalAveragePooling1D()(x)
    out = layers.Dense(1)(x)
    m = tf.keras.Model(inp, out, name='Conv1D_causal')
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='mse', metrics=['mae'])
    return m

ts_callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True),
            tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', patience=4, factor=.5)]

conv1d = build_conv1d()
conv1d.summary()
h1 = conv1d.fit(train_ts, validation_data=val_ts, epochs=80,
                callbacks=ts_callbacks, verbose=1)

## 11. Convoluciones dilatadas (`dilation_rate`)
En `Conv1D` dilatada, las posiciones del kernel están separadas por espacios. Con `kernel_size=3` y dilataciones `1, 2, 4, 9, 15`.
Así el modelo puede combinar información a diferentes escalas sin hacer una convolución con un kernel enorme. 

In [ ]:
DILATIONS = [1, 2, 4, 9, 15]
def build_dilated_conv1d():
    inp = tf.keras.Input(shape=(WINDOW_SIZE, 1))
    x = inp
    for rate in DILATIONS:
        x = layers.Conv1D(32, 3, padding='causal', dilation_rate=rate,
                          activation='relu', name=f'dilation_{rate}')(x)
    x = layers.GlobalAveragePooling1D()(x)
    out = layers.Dense(1)(x)
    m =tf.keras.Model(inp, out, name='Conv1D_dilatada')
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='mse', metrics=['mae'])
    return m

model_dilated = build_dilated_conv1d()
model_dilated.summary()
hd = model_dilated.fit(train_ts, validation_data=val_ts,
                       epochs=58, callbacks=ts_callbacks, verbose=1)

## 12. Comparación en test: Conv1D vs dilatada
Reconvertimos las predicciones a la escala original antes de calcular **MAE** y **RMSE**. Las gráficas respetan la posición temporal de cada objetivo, de modo que no hay errores de alineación.

In [ ]:
def evaluate_forecast(m, X, y, indices):
    pred_norm = m.predict(X, verbose=0).reshape(-1)
    actual = y.reshape(-1) * sigma + mu
    predicted = pred_norm * sigma + mu
    mae = np.mean(np.abs(actual - predicted))
    rmse = np.sqrt(np.mean((actual - predicted)**2))
    return predicted, actual, {'MAE': float(mae), 'RMSE': float(rmse)}

pred_1d, actual, metrics_1d = evaluate_forecast(conv1d, X_test, y_test, idx_test)
pred_dil, _, metrics_dil = evaluate_forecast(model_dilated, X_test, y_test, idx_test)
print(pd.DataFrame([metrics_1d, metrics_dil], index=['Conv1D', 'Conv1D dilatada']))
fig, ax = plt.subplots(figsize=(14,4))
ax.plot(dates.iloc[idx_test], actual, label='Observado', color='black', lw=1.5)
ax.plot(dates.iloc[idx_test], pred_1d, label='Conv1D', alpha=.8)
ax.plot(dates.iloc[idx_test], pred_dil, label='Conv1D dilatada', alpha=.8)
ax.set(xlabel='Fecha', ylabel='# manchas solares', title='Predicción en test')
ax.legend(); plt.tight_layout(); plt.show()